# Alakoro ML — Treinamento de CNN para Detecção de Eventos

Este notebook demonstra como treinar uma CNN 2D para classificar patches DAS com e sem evento.

In [ ]:
import numpy as np
import torch

from src.io.alakoro_spool import AlakoroPatch
from src.io.dasdae import DASDAEAdapter
from src.ml import DASDataset, EventCNN, Trainer, split_dataset

In [ ]:
# Dados sintéticos
patches, labels = [], []
for i in range(120):
    data = np.random.randn(64, 16).astype(np.float32)
    label = i % 2
    if label == 1:
        data[20:45, 5:11] += 4.0
    patch = DASDAEAdapter.array_to_patch(data, modality='das')
    patches.append(AlakoroPatch(patch, modality='das'))
    labels.append(label)

In [ ]:
dataset = DASDataset(patches, labels=labels, window_size=(32, 8), stride=(32, 8), normalize='zscore')
train, val, test = split_dataset(dataset)

train_loader = torch.utils.data.DataLoader(train, batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(val, batch_size=8)

In [ ]:
model = EventCNN(input_shape=(32, 8), n_classes=2)
trainer = Trainer(model, loss_fn=torch.nn.CrossEntropyLoss())
history = trainer.fit(train_loader, val_loader, epochs=10, early_stopping_patience=3)